# 01 — Bố trí thí nghiệm cho công bằng

Notebook 00 trả lời *thời gian nằm ở đâu*. Notebook này trả lời một câu khó hơn:
**làm sao biết con số mình đo được là thật?**

Trong lúc làm dự án này tôi mắc **bốn lỗi đo đạc**. Điểm chung của cả bốn: chúng
đều **làm kết quả đẹp lên**. Đó là đặc điểm nguy hiểm nhất của lỗi đo — nó thưởng
cho người không kiểm tra, nên không có động lực tự nhiên để tìm ra.

Notebook này dùng chính các file kết quả trong `results/`, không chạy lại mô hình.

In [1]:
import json, statistics
from pathlib import Path

def load(name):
    return json.loads(Path(f"../results/{name}").read_text())

print("các file kết quả có sẵn:")
for p in sorted(Path("../results").glob("*.json")):
    print(" ", p.name)

các file kết quả có sẵn:
  breakdown.json
  gate0_latency.json
  gate1_smolvlm2b.json
  gate2_confirm.json
  gate2_edge_sweep.json
  gate2_sweep.json
  gate3_quant.json
  gate4_serving_1536.json
  gate4_serving_768.json
  gate5_nf4.json
  preprocess_cost.json
  prune_smoke.json
  prune_smoke2.json
  smoke.json


## Lỗi 1 — vùng bấm giờ không công bằng

Khi so "đường chạy gốc" với "đường đã tối ưu", tôi vô tình để bộ mã hoá thị giác
**nằm ngoài vùng bấm giờ** ở đường tối ưu, vì nó được gọi sớm hơn để lấy token ảnh
ra cắt. Kết quả: một bên còn gánh nặng, một bên đã bỏ gánh.

Hai file dưới đây là cùng một thí nghiệm, chạy trước và sau khi sửa.

In [2]:
truoc = load("prune_smoke.json")    # trước khi sửa
sau   = load("prune_smoke2.json")   # sau khi sửa

for ten, r in [("TRƯỚC khi sửa", truoc), ("SAU khi sửa", sau)]:
    print(ten)
    for k, v in r["paired_speedup_vs_baseline"].items():
        print(f"   {k.split('(')[0]:<18} {v['median_speedup']:.2f}x")

TRƯỚC khi sửa
   keep0.5            3.47x
   keep0.25           4.36x
SAU khi sửa
   keep0.5            1.12x
   keep0.25           1.22x


Cùng một kỹ thuật, cùng một mô hình, cùng một máy: **3,47× trở thành 1,12×**.

Nếu không nghi ngờ ở chỗ "cắt một nửa token mà nhanh gấp 3,5 lần", cả dự án sẽ đi
tiếp trên một con số thổi phồng gấp ba.

**Quy tắc rút ra:** vùng bấm giờ phải bao trọn *toàn bộ* đường đi mà người dùng
thật phải trả, ở **mọi** cấu hình được so sánh.

## Lỗi 2 — mỗi vòng đo dùng một nhóm mẫu khác nhau

Ban đầu tôi chia tập mẫu thành các nhóm, mỗi vòng đo một nhóm. Nghe hợp lý, nhưng
ảnh trong ChartQA có kích thước rất khác nhau, nên số token ảnh dao động mạnh.
Hệ quả: **phần lớn dao động đến từ việc mẫu này nặng hơn mẫu kia**, chứ không đến
từ hệ thống.

Triệu chứng: khoảng tứ phân vị lên tới 86,5% trung vị, và "độ trôi theo thời gian"
ra **−44%**, tức máy càng chạy càng nhanh — điều vô lý.

In [3]:
# Vì sao so sánh theo cặp lại miễn nhiễm với việc mẫu nặng nhẹ khác nhau
mau = {"ảnh nhỏ": 100.0, "ảnh vừa": 400.0, "ảnh lớn": 1000.0}
toi_uu = {k: v / 2 for k, v in mau.items()}          # kỹ thuật làm nhanh gấp đôi

# Cách sai: mỗi vòng bốc một nhóm mẫu khác nhau rồi so hai trung vị
goc_nhom_A = statistics.median([mau["ảnh nhỏ"], mau["ảnh vừa"]])
toiuu_nhom_B = statistics.median([toi_uu["ảnh vừa"], toi_uu["ảnh lớn"]])
print(f"so hai nhóm khác nhau : {goc_nhom_A / toiuu_nhom_B:.2f}x  (sai)")

# Cách đúng: với TỪNG mẫu, lấy tỉ số rồi mới tổng hợp
ti_so = [mau[k] / toi_uu[k] for k in mau]
print(f"so theo cặp từng mẫu  : {statistics.median(ti_so):.2f}x  (đúng)")

so hai nhóm khác nhau : 0.71x  (sai)
so theo cặp từng mẫu  : 2.00x  (đúng)


Cách sai cho **0,71×** — tức là kết luận rằng kỹ thuật tối ưu làm mọi thứ **chậm đi**,
trong khi nó thực sự nhanh đúng **2,00×**. Toàn bộ sai lệch đến từ việc hai nhóm mẫu
có độ nặng khác nhau, không đến từ kỹ thuật.

**Quy tắc rút ra:** các vòng đo phải là **bản lặp trên cùng tập mẫu**, và mọi so
sánh phải làm theo cặp trên từng mẫu.

## Lỗi 3 — kết luận "không khác biệt" khi chưa đủ mẫu

Đây là lỗi tinh vi nhất, vì nó trông giống một kết luận khoa học cẩn trọng.

Với 100 mẫu, cấu hình `edge768` cho độ chính xác thấp hơn bản gốc 6 điểm, nhưng
phép kiểm cho p = 0,18 — "không phân biệt được". Rất dễ đọc thành "giảm độ phân
giải không làm mất chất lượng".

Tôi chạy lại với 300 mẫu.

In [4]:
import sys
sys.path.insert(0, "..")
from bench.metrics import paired_accuracy

for ten, f in [("100 mẫu", "gate2_edge_sweep.json"), ("300 mẫu", "gate2_confirm.json")]:
    r = load(f)
    pa = paired_accuracy(r["records"], "baseline", "edge768(edge=768)")
    acc_goc = 100 * r["configs"]["baseline"]["accuracy"]
    acc_moi = 100 * r["configs"]["edge768(edge=768)"]["accuracy"]
    print(f"{ten}: {acc_goc:.1f}% -> {acc_moi:.1f}% ({acc_moi-acc_goc:+.1f} điểm) | "
          f"cặp bất đồng {pa['only_a_correct']}+{pa['only_b_correct']}="
          f"{pa['only_a_correct']+pa['only_b_correct']} | p = {pa['p_value']:.4f}")

100 mẫu: 69.0% -> 63.0% (-6.0 điểm) | cặp bất đồng 10+4=14 | p = 0.1796
300 mẫu: 64.7% -> 57.3% (-7.3 điểm) | cặp bất đồng 35+13=48 | p = 0.0021


Mức giảm gần như không đổi (6,0 rồi 7,3 điểm), nhưng **số cặp bất đồng tăng từ 14
lên 48**, và kết luận đảo ngược: từ "không phân biệt được" thành "giảm rõ rệt".

**Ký hiệu mới — kiểm định McNemar.** Khi hai cấu hình chạy trên cùng tập mẫu, các
mẫu mà cả hai cùng đúng hoặc cùng sai **không mang thông tin so sánh**. Chỉ các
*cặp bất đồng* mới nói lên điều gì. Gọi b là số mẫu chỉ cấu hình A đúng, c là số
mẫu chỉ cấu hình B đúng. Nếu hai cấu hình ngang nhau thì mỗi cặp bất đồng giống
như tung một đồng xu công bằng, nên:

    p = xác suất thấy chênh lệch ít nhất bằng |b − c| khi tung b+c lần đồng xu cân

Vì phép kiểm chỉ dùng b và c, **lực của nó phụ thuộc vào số cặp bất đồng**, không
phụ thuộc tổng số mẫu. Đó là lý do 100 mẫu không đủ: chỉ có 14 cặp để dựa vào.

**Quy tắc rút ra:** "p lớn" nghĩa là *chưa đủ bằng chứng*, không phải *không có
khác biệt*. Trước khi kết luận, hãy đếm xem mình có bao nhiêu cặp bất đồng.

## Lỗi 4 — giá trị mặc định lặng lẽ đổi thí nghiệm

Tôi chạy một phép đo mà quên truyền `--model`, nên bộ đo dùng mặc định là mô hình
**256 triệu tham số** thay vì bản **2,2 tỷ** dùng xuyên suốt dự án. Kết quả trông
như một thảm hoạ kỹ thuật: độ chính xác tụt từ 64% xuống 23%, số token ảnh đổi từ
1.053 thành 640, độ trễ dao động trên 80%.

Tôi suýt kết luận rằng lượng tử hoá 4-bit phá hỏng mô hình.

Dấu hiệu lẽ ra phải nhận ra ngay: **số token ảnh đổi**. Token ảnh là hàm của bộ xử
lý ảnh và kiến trúc, không phải của kiểu số. Một đại lượng lẽ ra bất biến mà lại
đổi thì tức là mình đang đo một thứ khác với thứ mình nghĩ.

**Quy tắc rút ra:** mỗi lần đo phải ghi lại *đại lượng bất biến* — ở đây là số
token ảnh và tên mô hình — và kiểm tra chúng trước khi đọc kết quả.

In [5]:
# Bảng đối chứng: các đại lượng phải bất biến giữa các lần chạy cùng mô hình
for f in ["gate2_edge_sweep.json", "gate2_confirm.json", "gate5_nf4.json"]:
    try:
        r = load(f)
    except FileNotFoundError:
        continue
    c = r["configs"]["baseline"]
    print(f"{f:<26} mô hình {r['model'].split('/')[-1]:<18} "
          f"kiểu số {r.get('quant','bf16'):<5} token ảnh {c['image_tokens_median']:.0f}")

gate2_edge_sweep.json      mô hình SmolVLM-Instruct   kiểu số bf16  token ảnh 1053
gate2_confirm.json         mô hình SmolVLM-Instruct   kiểu số bf16  token ảnh 1053
gate5_nf4.json             mô hình SmolVLM-Instruct   kiểu số nf4   token ảnh 1053


## Danh sách kiểm tra rút ra

Sáu quy tắc dưới đây được cài thẳng vào `bench/harness.py`, không để ở dạng ghi chú:

1. Vùng bấm giờ bao trọn đường đi thật, ở mọi cấu hình
2. Các vòng đo là bản lặp trên cùng tập mẫu
3. Xen kẽ và ngẫu nhiên hoá thứ tự các cấu hình, có seed để tái lập
4. So sánh theo cặp trên từng mẫu, báo cáo trung vị và khoảng tứ phân vị
5. Chỉ tuyên bố cải thiện khi vượt ba lần nhiễu đã đo (ở máy này là 25,5%)
6. Ghi kèm đại lượng bất biến và trạng thái máy cho từng phép đo

Và một quy tắc không cài được vào mã, chỉ nằm ở thói quen: **khi một kết quả đẹp
bất ngờ, hãy nghi ngờ phép đo trước khi mừng.** Ba trong bốn lỗi ở trên được phát
hiện đúng theo cách đó.